# Precision and devices

PyTorch users know this runtime error:

```python
torch.randn(4, dtype=torch.float64, device="mps")
# TypeError: Cannot convert a MPS Tensor to float64 dtype as the MPS
# framework doesn't support float64. Please use float32 instead.
```

MLX dropped float64 on Metal in 0.31, and you find out at op-launch — deep in a forward
pass. idris-ml's `Tensor` carries both a **device** and a **dtype** at the type level, and
a `Compatible (ex, dt)` capability check makes (Metal GPU, float64) a compile error.

## The four-parameter Tensor

```idris
record Tensor (dims : Vect rank Nat) (0 ex : Executor) (0 dt : DType) (0 g : GradMode) where
  constructor MkTensor
  tensorPtr : AnyPtr
  paramId   : Maybe String
```

- `dims` — the shape (the usual dependently-typed `Vect`).
- `ex` — the executor: `TapeExecutor`, `TorchExecutor d`, `MlxExecutor s`, or your own.
- `dt` — the dtype: `F32`, `F64`, `BF16`, `F16`, `IntN n`, `UInt n`, `Bool`.
- `g` — the grad mode (`WithGrad` / `NoGrad`).

All four are erased at runtime (the `0` quantity) — they exist only to let the compiler
prove things before the program runs.

## The `Compatible` table

Which (device, dtype) pairs are allowed? An empty marker interface — the existence of an
instance IS the proof:

```idris
interface Compatible (0 ex : Executor) (0 t : DType) where
```

| Executor          | F64 | F32 |
|-------------------|-----|-----|
| `TapeExecutor`    | ✓   | ✓   |
| `TorchExecutor TCpu` | ✓ | ✓ |
| `TorchExecutor TMps` |   | ✓ |
| `MlxExecutor MCpu`| ✓   | ✓   |
| `MlxExecutor MGpu`|     | ✓   |

The two deliberately missing cells — `(MlxExecutor MGpu, F64)` and
`(TorchExecutor TMps, F64)` — are where the demo errors live.

In [ ]:
:doc Compatible

We isolate the dtype axis with a tiny witness that needs only `Compatible` (no construction, so it's build-independent):

In [ ]:
compatOK : Compatible ex dt => ()
compatOK = ()

### Positive case

MLX GPU at F32 is admissible — `Compatible (MlxExecutor MGpu) F32` exists:

In [ ]:
:exec putStrLn (show (compatOK {ex=MlxExecutor MGpu} {dt=F32}))

### Negative case

F64 on the same device has no instance, so it's a compile error (*expected to fail*):

In [ ]:
:exec putStrLn (show (compatOK {ex=MlxExecutor MGpu} {dt=F64}))

## Parametric dtype families

The dtype tags are `Nat`-parameterized type constructors, not opaque names:

```idris
data Float : Nat -> Type where MkFloat : Float n
F32 : Type ; F32 = Float 32
F64 : Type ; F64 = Float 64
```

Four families plus `Bool`: `Float n` (IEEE), `BFloat n` (brain-float, `BF16 = BFloat 16`),
`IntN n` (signed — `Int` is reserved, hence the `N`), `UInt n` (unsigned).

In [ ]:
:t Float 64
:t F32
:t IntN 32
:doc IsDType

## `UpcastableTo` — derived lossless conversion

Lossless conversion is derived from an `LTE m n` proof on bit-widths: within a family
(`Float 32 → Float 64`, `IntN 8 → IntN 64`), and across the float families when both the
mantissa and exponent fit (`BF16 → F32` is lossless — BF16's 7 mantissa bits ≤ F32's 23,
same 8-bit exponent). Idris's auto-search synthesises the proof from the `Nat`
constructors:

```idris
LTE (mantissaBits {t=from}) (mantissaBits {t=to}) =>
LTE (exponentBits {t=from}) (exponentBits {t=to}) =>
LosslessTo from to where    -- bridges to UpcastableTo
```

In [ ]:
-- F32 -> F64 is lossless (LTE 32 64 provable)
the (UpcastableTo F32 F64) %search
-- I8 -> I64 is lossless within the signed-int family
the (UpcastableTo I8 I64) %search
-- BF16 -> F32 is lossless across families (mantissa 7 <= 23, exponent 8 = 8)
the (UpcastableTo BF16 F32) %search

Narrowing conversions, and ones that change representation (float ↔ int), have no `UpcastableTo` instance — these are *expected to fail*:

In [ ]:
-- F64 -> F32 is narrowing (LTE 64 32 has no proof)
the (UpcastableTo F64 F32) %search

In [ ]:
-- F32 -> IntN 32 changes representation (float to int)
the (UpcastableTo F32 (IntN 32)) %search

## `tcast` and `tcastUnsafe`

```idris
tcast       : (UpcastableTo from to, IsDType from, IsDType to) =>
              Tensor dims ex from g -> IO (Tensor dims ex to g)
tcastUnsafe : (0 to : DType) -> (IsDType from, IsDType to) =>
              Tensor dims ex from g -> IO (Tensor dims ex to g)
```

`tcast` is the safe upcast — the compiler verifies via `UpcastableTo` that no information
is lost. `tcastUnsafe` is for deliberately narrowing (F64 → F32) or cross-family casts; the
`Unsafe` suffix and explicit target dtype are the signal that the caller takes
responsibility for the precision loss (the lossy edge stays code-visible, never hidden in
an autocast context).

## Build-mode targeting

Idris-2 can't drive type-level selection from a runtime env var — types fix at elaboration.
So the Makefile reads `BACKEND` + `MLX_DEVICE` + `TORCH_DEVICE` and emits `BuildConfig.idr`:

```idris
public export
ExampleDevice : Type
ExampleDevice = TapeExecutor       -- or MlxExecutor MGpu in F32 mode

public export
ExampleDType : DType
ExampleDType = F64                 -- or F32 in F32 mode
```

Examples reference `ExampleDevice` / `ExampleDType` (aliased `Ex` / `F`) instead of
hardcoding, so switching modes is a Makefile flip — no source edits:

```bash
make BACKEND=tape install                  # F64 on the tape backend (default)
make BACKEND=mlx MLX_DEVICE=gpu install    # F32 on Metal GPU
```

## Summary

| | PyTorch | idris-ml |
|---|---------|----------|
| Dtype tracking | runtime (`.dtype`) | compile-time (phantom `dt`) |
| Device-dtype mismatch | `RuntimeError`/`TypeError` at op launch | type error before compilation |
| Lossless upcast | implicit, sometimes silent | `tcast` (verified via `UpcastableTo`) |
| Narrowing / cross-family | implicit, no warning | explicit `tcastUnsafe` |
| Multi-mode targeting | code change | build flag |

Next: [01 Tensors and Types](01_tensors_and_types.ipynb) · [06 Device Safety](06_device_safety.ipynb).